# CHSM8, nano: train a chess model in plain Python

A toy version of CHSM8 pretraining, small enough for a laptop CPU (a few minutes). Everything below is ordinary
Python: the data, the batches, the training loop and the games are written out in these cells, on top of the same
modules our real runs use:

- `src/factored_tokenizer.py`: a move becomes one row `(kind, src, dst, piece, promo)`
- `src/model.py`: `FactoredChessDecoder` (the CHSM8 decoder) and `factored_cross_entropy` (its loss)
- `src/train_pretrain.py`: `build_block_causal_mask`, so packed games cannot see each other

The real run is the same model at depth 12, width 960, on 1.07B games (`recipes/1_pretrain.sh`).
SFT, DPO and the player models are in `recipes/`, one script per stage.

In [ ]:
import os, sys, time, random
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import chess
import numpy as np
import torch
import pyarrow.parquet as pq
from huggingface_hub import snapshot_download

from factored_tokenizer import move_to_row, ROW_BOS, ROW_EOS, ROW_PAD, KIND_MOVE, PIECE_TO_IDX, PROMOTION_TO_IDX
from model import DecoderConfig, FactoredChessDecoder, factored_cross_entropy
from train_pretrain import build_block_causal_mask

torch.manual_seed(0); random.seed(0)
torch.set_num_threads(min(8, os.cpu_count()))
MONTH, N_GAMES = "2014-01", 5000          # one month of Lichess; raise N_GAMES for a better model

## 1. Games
`N_GAMES` games from [chsm8-pre-training](https://huggingface.co/datasets/LegumMagister/chsm8-pre-training): filtered
Lichess games with their moves in UCI. We keep the first 90% for training and the rest for validation.

In [ ]:
path = snapshot_download("LegumMagister/chsm8-pre-training", repo_type="dataset",
                         allow_patterns=[f"data/{MONTH}/part-00000.parquet"])
games = pq.read_table(f"{path}/data/{MONTH}/part-00000.parquet", columns=["white_elo", "black_elo", "result", "moves"])
games = games.slice(0, N_GAMES).to_pylist()
print(len(games), "games, e.g.:", games[0]["result"], games[0]["moves"][:80], "...")

## 2. Moves to rows
Each game becomes `BOS, move, move, ..., EOS`. A move is a row `(kind, src, dst, piece, promo)`: from-square,
to-square, piece type and promotion, read off the board with `move_to_row`. The model predicts each of the five parts
with its own small head, instead of picking from one big vocabulary of move strings.

In [ ]:
def game_rows(uci_moves):
    board, rows = chess.Board(), [ROW_BOS]
    for u in uci_moves.split():
        move = chess.Move.from_uci(u)
        rows.append(move_to_row(move, board))
        board.push(move)
    return rows + [ROW_EOS]

encoded = [game_rows(g["moves"]) for g in games]
print("first moves of game 0:", encoded[0][:4])
print(f"{sum(map(len, encoded)):,} rows in {len(encoded):,} games")

## 3. Packing
Training sequences are 512 rows long. Games are laid back to back and cut into records of 513 rows: input rows
`[:-1]`, target rows `[1:]` (predict the next row). Several games share a record; the block-causal mask built in
the training step keeps each game from attending to the others. This is the packed format of `prep/pretraining/pack.py`.

In [ ]:
SEQ = 512

def pack(game_list):
    stream = [r for g in game_list for r in g]
    n = len(stream) // (SEQ + 1)
    return torch.tensor(stream[: n * (SEQ + 1)], dtype=torch.long).view(n, SEQ + 1, 5)

split = int(0.9 * len(encoded))
train, val = pack(encoded[:split]), pack(encoded[split:])
print("train records:", tuple(train.shape), " val records:", tuple(val.shape))

## 4. The model
`FactoredChessDecoder`: a Llama-style decoder (RoPE, RMSNorm, SwiGLU, QK-norm) with factorized embeddings and heads.
Here 4 layers of width 128, about 1M parameters (CHSM8: 12 layers of width 960, 210M).

In [ ]:
cfg = DecoderConfig(vocab_size=0, d_model=128, n_heads=4, n_layers=4, d_ff=512, max_seq_len=SEQ, dropout=0.0,
                    architecture="llama", llama_qk_norm=True, use_cross_attention=False)
model = FactoredChessDecoder(cfg)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 5. Training
Plain AdamW with warmup and cosine decay. The loss is `factored_cross_entropy`: cross-entropy on `kind` everywhere,
plus on `src`, `dst`, `piece` and `promo` at move positions.

In [ ]:
STEPS, BATCH, LR = 600, 8, 2e-3
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))
lr_at = lambda s: LR * min(1, (s + 1) / 30) * 0.5 * (1 + np.cos(np.pi * s / STEPS))

def loss_on(batch):
    x, y = batch[:, :-1], batch[:, 1:]
    return factored_cross_entropy(model(x, mask=build_block_causal_mask(x)), y)

@torch.no_grad()
def val_loss():
    model.eval()
    l = np.mean([loss_on(val[i:i + BATCH]).item() for i in range(0, len(val), BATCH)])
    model.train()
    return l

t0 = time.time()
model.train()
for step in range(STEPS):
    for g in opt.param_groups:
        g["lr"] = lr_at(step)
    loss = loss_on(train[torch.randint(len(train), (BATCH,))])
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 100 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  train {loss.item():.3f}  val {val_loss():.3f}  {time.time() - t0:.0f}s")

## 6. Playing
To move, the model reads the game so far and scores every **legal** move: the log-probability of a move is the sum
of its `src`, `dst`, `piece` and `promo` log-probabilities. It plays the best one (or samples, with a temperature).
No search, and illegal moves are impossible by construction.

In [ ]:
@torch.no_grad()
def move_scores(board):
    rows = [ROW_BOS]
    replay = chess.Board()
    for m in board.move_stack:
        rows.append(move_to_row(m, replay)); replay.push(m)
    x = torch.tensor([rows[-SEQ:]])
    model.eval()
    lp = {k: v[0, -1].log_softmax(-1) for k, v in model(x).items()}
    return {m: (lp["src"][m.from_square] + lp["dst"][m.to_square]
                + lp["piece"][PIECE_TO_IDX[board.piece_at(m.from_square).piece_type]]
                + lp["promo"][PROMOTION_TO_IDX[m.promotion]]).item()
            for m in board.legal_moves}

def model_move(board, temperature=0.0):
    s = move_scores(board)
    moves = list(s)
    if temperature == 0:
        return max(moves, key=s.get)
    p = torch.tensor([s[m] for m in moves]).div(temperature).softmax(0)
    return moves[torch.multinomial(p, 1).item()]

board = chess.Board()
for _ in range(10):
    board.push(model_move(board))
print("the model's opening:", chess.Board().variation_san(board.move_stack))
s = move_scores(board)
p = torch.tensor(list(s.values())).softmax(0)                  # probabilities over the legal moves
top = sorted(zip(s, p.tolist()), key=lambda kv: -kv[1])[:5]
print("its top replies here:", [(board.san(m), f"{q:.0%}") for m, q in top])

## 7. A small match
The nano model against a player that moves at random, 200 plies at most. After a few minutes of training it knows
openings and rarely loses, but it cannot yet finish a game: expect draws and unfinished games (`*`), not wins. More
games and steps fix that; the real CHSM8 plays at about 1,800 Elo against Stockfish. With `STOCKFISH` set to a
binary, the nano model also plays Stockfish at its lowest level.

In [ ]:
def play(white, black, max_plies=200):
    board = chess.Board()
    while not board.is_game_over(claim_draw=True) and board.ply() < max_plies:
        board.push((white if board.turn else black)(board))
    return board.result(claim_draw=True)

rand = lambda b: random.choice(list(b.legal_moves))
nano = lambda b: model_move(b, temperature=0.5)
FLIP = {"1-0": "0-1", "0-1": "1-0", "1/2-1/2": "1/2-1/2", "*": "*"}         # nano's result when it plays Black
results = [play(nano, rand) for _ in range(5)] + [FLIP[play(rand, nano)] for _ in range(5)]
print("nano vs random (nano's score first):", {r: results.count(r) for r in set(results)})

if os.environ.get("STOCKFISH"):
    import chess.engine
    with chess.engine.SimpleEngine.popen_uci(os.environ["STOCKFISH"]) as sf:
        sf.configure({"Skill Level": 0})
        fish = lambda b: sf.play(b, chess.engine.Limit(time=0.01)).move
        print("nano vs Stockfish (skill 0):", [play(nano, fish), FLIP[play(fish, nano)]])